## 06 · Living With Tomorrow

Notebook 05 showed that the AI learned a routine that only works on days starting with a 70% battery, because that's the only kind of day it ever practised on. This notebook changes how it practises: instead of single days that reset at midnight, it trains on continuous weeks, so the battery it drains tonight is the battery it has tomorrow.

Same grid, same scoring rule, same training settings. Only the length of a practice run changes.

In [4]:
# Load tools and train the agent on continuous weeks (or load it if already trained)
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from minigrid_mdp.env.week_env import WeekLongEnv
from minigrid_mdp.agents import RuleBasedAgent, DRLAgent
from minigrid_mdp.metrics import run_weeks, steady_state_days, summarise, compare
from minigrid_mdp.training import train_or_load

MODELS_DIR = Path("..") / "models"
WEEK_MODEL_PATH = MODELS_DIR / "ppo_week_trained.zip"

train_or_load(WeekLongEnv, WEEK_MODEL_PATH)

### The continuous-week test, again

Same test as notebook 05: 4 continuous weeks, ignoring the first day of each. Three operators: the rulebook, the AI from notebook 04 (trained on single days), and the new AI trained on weeks.

In [5]:
# Run all three operators for 4 continuous weeks and compare steady-state days
week_env = WeekLongEnv()
operators = {
    "rulebook":        RuleBasedAgent(week_env.action_space),
    "ai_day_trained":  DRLAgent(week_env.action_space, model_path=str(MODELS_DIR / "ppo_fixed_reward.zip")),
    "ai_week_trained": DRLAgent(week_env.action_space, model_path=str(WEEK_MODEL_PATH)),
}
week_logs = {name: steady_state_days(run_weeks(week_env, agent)) for name, agent in operators.items()}
compare({name: summarise(log) for name, log in week_logs.items()})

rulebook       ai_day_trained        ai_week_trained      
                     mean   std           mean    std            mean   std
diesel_hours_pct    47.40  2.40          27.95   2.60           42.53  1.73
fuel_cost_usd       39.30  1.89          29.95   2.59           36.86  2.32
unmet_pct            0.31  0.69           5.94   2.30            0.36  0.75
end_soc              0.34  0.07           0.36   0.12            0.37  0.12
reward             -42.10  5.94         -77.77  18.94          -40.31  6.54

**What this shows:** trained on continuous weeks, the AI no longer runs the battery dry before dawn. It serves the village about as reliably as the rulebook (0.36% vs 0.31% unmet) and spends about 6% less on fuel: $36.86 a day against $39.30. On the full score it now comes out slightly ahead of the rulebook, where the day-trained AI fell far behind.

In [6]:
# Compare fuel cost day by day: on how many of the 24 days was the week-trained AI cheaper?
daily = pd.DataFrame({name: summarise(week_logs[name]).fuel_cost_usd for name in ["rulebook", "ai_week_trained"]})
saving = daily.rulebook - daily.ai_week_trained
print(f"AI cheaper on {(saving > 0).sum()} of {len(saving)} days | daily saving: mean ${saving.mean():.2f}, min ${saving.min():.2f}, max ${saving.max():.2f}")

AI cheaper on 19 of 24 days | daily saving: mean $2.44, min $-3.78, max $7.25


**Is the saving real?** Both operators faced the same 24 days. The AI was cheaper on 19 of them, saving $2.44 a day on average, between $7.25 better and $3.78 worse. Winning 19 of 24 by chance would happen less than 1% of the time, so the saving is consistent rather than lucky. It's still modest, and some days the rulebook does better.

## Summary

- Training on continuous weeks fixed the pre-dawn failure: the AI now has to live with the battery it leaves behind, and it learned to.
- In a world that never resets at midnight, it saves **about 6% on fuel at the same reliability** as the rulebook, cheaper on 19 of 24 days.
- That's far smaller than the 26% originally reported. Each step of this series removed a source of false improvement: a grid with almost no sun, energy that appeared from nowhere, a scoring rule that made cutting power free, and a simulation that forgave every battery debt at midnight.
- What's left is small, but it's real, and we know exactly where it comes from.